In [13]:
import pandas as pd
import numpy as np
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

In [14]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Specify the exact file name within the dataset
file_path = "IMDB Dataset.csv"

# Use dataset_load to resolve the DeprecationWarning
df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "lakshmi25npathi/imdb-dataset-of-50k-movie-reviews",
  file_path,
)

print("First 5 records:\n", df.head())

Using Colab cache for faster access to the 'imdb-dataset-of-50k-movie-reviews' dataset.
First 5 records:
                                               review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive


In [15]:
df.shape

(50000, 2)

In [16]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB


In [17]:
df["sentiment"].value_counts()

,count
sentiment,
positive,25000
negative,25000


In [18]:
from sklearn.model_selection import train_test_split

X = df["review"]
y = df["sentiment"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [19]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((40000,), (10000,), (40000,), (10000,))

In [20]:
y_train.head()

,sentiment
39087,negative
30893,negative
45278,positive
16398,negative
13653,negative


In [21]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

In [22]:
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

y_train, y_test

(array([0, 0, 1, ..., 0, 1, 1]), array([1, 1, 0, ..., 1, 0, 1]))

In [23]:
le.classes_, le.transform(le.classes_)

(array(['negative', 'positive'], dtype=object), array([0, 1]))

In [24]:
import joblib
joblib.dump(le, 'le.joblib')

['le.joblib']

In [25]:
word_counts = X_train.str.split().str.len()

word_counts.describe()

,review
count,40000.000000
mean,231.006425
std,171.612227
min,4.000000
25%,126.000000
50%,173.000000
75%,280.000000
max,2470.000000


In [26]:
word_counts.quantile(0.90)

np.float64(450.0)

In [27]:
from tensorflow.keras.preprocessing.text import one_hot
from tensorflow.keras.preprocessing.sequence import pad_sequences

max_features = 10000
max_sequence_len = 450

X_train= [one_hot(text, n=max_features) for text in X_train]
X_test = [one_hot(text, n=max_features) for text in X_test]


X_train = pad_sequences(
    X_train,
    maxlen=max_sequence_len,
    padding='pre',
    truncating='pre'
)

X_test = pad_sequences(
    X_test,
    maxlen=max_sequence_len,
    padding="pre",
    truncating="pre"
)

In [28]:
np.savez_compressed('X_data.npz', X_train=X_train, X_test=X_test)
np.savez_compressed('y_data.npz', y_train=y_train, y_test=y_test)

In [72]:
X_loaded_data = np.load('X_data.npz')

X_train = X_loaded_data['X_train']
X_test = X_loaded_data['X_test']

In [73]:
y_loaded_data = np.load("y_data.npz")

y_train = y_loaded_data["y_train"]
y_test = y_loaded_data["y_test"]

In [74]:
model = Sequential([
    Embedding(max_features, 64),
    SimpleRNN(64),
    Dense(1, activation="sigmoid")
])

In [75]:
model.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_4 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_4 (SimpleRNN)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [81]:
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime

current_time = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
dynamic_log_dir = f'logs/run_{current_time}'

early_stopping = EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)
tensorboard = TensorBoard(
    log_dir=dynamic_log_dir,
    histogram_freq=1,
)


In [82]:
from tensorflow.keras.optimizers import Adam

optimizer = Adam(learning_rate=0.0001)

In [83]:
from tensorflow.keras.metrics import Precision, Recall

model.compile(
    optimizer = optimizer,
    loss = "binary_crossentropy",
    metrics=[
        "accuracy",
        Precision(name='precision'),
        Recall(name='recall'),
        ]
)

In [84]:
print(np.unique(y_train))

[0 1]


In [85]:
current_time = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
dynamic_log_dir = f'logs/run_{current_time}'

model.fit(
    X_train,
    y_train,
    epochs=10,
    validation_split=0.2,
    batch_size=32,
    callbacks=[early_stopping, tensorboard]
)

Epoch 1/10
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 117s 116ms/step - accuracy: 0.8526 - loss: 0.3557 - precision: 0.8453 - recall: 0.8623 - val_accuracy: 0.7965 - val_loss: 0.4534 - val_precision: 0.7933 - val_recall: 0.8020
Epoch 2/10
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 140s 114ms/step - accuracy: 0.8640 - loss: 0.3292 - precision: 0.8632 - recall: 0.8644 - val_accuracy: 0.8054 - val_loss: 0.4405 - val_precision: 0.8202 - val_recall: 0.7822
Epoch 3/10
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 116s 116ms/step - accuracy: 0.8729 - loss: 0.3156 - precision: 0.8711 - recall: 0.8746 - val_accuracy: 0.8094 - val_loss: 0.4358 - val_precision: 0.8101 - val_recall: 0.8083
Epoch 4/10
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 146s 120ms/step - accuracy: 0.8824 - loss: 0.2968 - precision: 0.8823 - recall: 0.8818 - val_accuracy: 0.8134 - val_loss: 0.4309 - val_precision: 0.8033 - val_recall: 0.8300
Epoch 5/10
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 136s 114ms/step - accuracy: 0.8535 - loss: 0.4190 - precision: 0.8901 - recall: 0.8057 - val_a

In [86]:
model.save("rnn_imdb.h5")

In [90]:
from sklearn.metrics import accuracy_score, precision_score, recall_score

y_pred_prob = model.predict(X_test)
y_pred_classes = (y_pred_prob > 0.5).astype(int)

accuracy = accuracy_score(y_test, y_pred_classes)
precision = precision_score(y_test, y_pred_classes)
recall = recall_score(y_test, y_pred_classes)

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")

313/313 ━━━━━━━━━━━━━━━━━━━━ 11s 35ms/step
Accuracy:  0.8277
Precision: 0.8206
Recall:    0.8422


In [106]:
y_pred_prob

array([[0.6450573 ],
       [0.5288515 ],
       [0.14221528],
       ...,
       [0.87755704],
       [0.01497769],
       [0.92558587]], dtype=float32)

In [87]:
model.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_4 (Embedding)         │ (32, 450, 64)          │       640,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_4 (SimpleRNN)        │ (32, 64)               │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (32, 1)                │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,944,965 (7.42 MB)

 Trainable params: 648,321 (2.47 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 1,296,644 (4.95 MB)

In [89]:
from tensorflow.keras.preprocessing.text import one_hot
from tensorflow.keras.preprocessing.sequence import pad_sequences

max_features = 10000
max_sequence_len = 450

def preprocess_text(text):
  text = [one_hot(text, n=max_features)]
  text = pad_sequences(
      text,
      maxlen=max_sequence_len,
      padding="pre",
      truncating="pre"
  )

  return text

In [95]:
def predict_sentiment(text):
  text = preprocess_text(text)
  y_pred_prob = model.predict(text)

  sentiment = "Positive" if y_pred_prob[0][0] > 0.5 else "Negative"

  return sentiment, y_pred_prob[0][0]

In [112]:
trial_review = """
the first half of the movie is very sloppy and slow. the songs of the
movie are superb but doesn't move with the soul of the movie , you will
feel that the songs are too fast and hip hop and the movie is very long
and slow moving.The actual story of the script starts from the second
half, where many logic is being questioned.the cinematography of the
movie is not up to the mark,it makes you feel that your watching 2
different movies.
"""
sentiment, score = predict_sentiment(trial_review)

sentiment, score

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step


('Negative', np.float32(0.2611558))